This is a simulation to simulate a $\phi^4$ on the Lattice via standard lattice computation techniques. The lagrangian for a standard $\phi^4$ theory is $$ \mathcal{L} = \frac{1}{2} (\partial_\mu \phi)^2 - \frac{1}{2}m^2 \phi^2 - \frac{\lambda}{4!} \phi^4 $$ which would yield a physical action of $$ S = \int d^4 x ~ \mathcal{L} = \int d^4 x \left( \frac{1}{2} (\partial_\mu \phi)^2 - \frac{1}{2} m^2 \phi^2 - \frac{\lambda}{4!} \phi^4 \right)$$The (euclideanized) action for a $\phi^4$ theory is $$ S_E = \sum_{x} \left( \sum_{\mu}\frac{1}{2} \left( \phi_{x+\hat{\mu}} - \phi_{x}  \right)^2  + \frac{m_0^2}{2} \phi_x^2 + \frac{\lambda}{4!} \phi_x^4  \right) $$ where the $x$ sum is all lattice points and the $\mu$ sum is the sum over all directions ($x,y,z,it,\text{etc}$). We intend to extract the physical mass of the particle over a sufficient amount of time by calculating the correlation function  $$C \equiv \langle \phi(x) \phi(0) \rangle = \frac{\int \mathcal{D}^N \phi ~ \langle \phi(x) \phi(0) \rangle  e^{-S_E}}{Z} $$ Where we define  $$ Z \equiv  \int \mathcal{D}^N \phi ~ e^{-S_E} $$. Extracting the mass of the particle functions via a large time limit of this project. In the large time limit $$ C  \sim A( e^{- m t} +  e^{-m(T-t)})  $$ so we can have $m_R (t) \sim \cosh^{-1}\left( \frac{C(t-1)+C(t+1)}{2 C(t)}\right)$


Pseudocode:

import all packages

let $L$ = edge length of lattice ($L \times L$),

let $\phi = [L \times L ~ \text{array}]$ = our scalar field

let $n = \# \text{sweeps}$

let $\epsilon = \text{random number}$

- At this point im pretty sure we don't need to define any other variables. Now we can continue with the algorithm
- The algorithm is a _Markov-Chain Monte Carlo_ algorithm where we choose to accept or reject a change based on its probability of happening (metropolis step)

1. Initialize all of our $\phi_x$ and $\phi_y$ values randomly
2. Compute change:

- For $x \in \{0,1,\dots,L-1\}$
  - For $y \in \{0,1,\dots,L-1\}$ 
     - let $\Delta \phi \in [-\epsilon,\epsilon]$
        - $\phi(x,y) \to \phi(x,y) + \Delta \phi$
        - ~~Compute plaquettes~~
        - Calculate $\Delta S_E$
        - Let $u \in [0,1]$
        - If $e^{- \Delta S_E} \ge u$
          - Accept
        - If $e^{- \Delta S_E} < u$
          - Reject; revert back to old value 
4. Compute $$ \langle O(\phi)\rangle \sim \frac{1}{M}   \sum_{i=1}^M O[\phi^{(i)}]$$ where $$ O[\phi] = \phi (x) \phi(0) $$ for various sample field configs $\phi^{(i)}$ (at $\mathbf{p} = 0$) and we have the mass of the particle based on our theory from before


In [3]:
# cold start
import matplotlib.pyplot as plt
import numpy as np
import time
L = 64
sweeps = 1000
m2 = 0.4
lam = 1.0
epsilon = 1.2
phi = np.random.uniform(-1,1, size =(L,L))
phi = np.zeros((L,L))

In [4]:
def sweep_metropolis(phi,m2,lam,epsilon):
    accepted = 0
    for i in range(L):
        for j in range(L):
            old_val = phi[i, j]
            new_val = old_val + np.random.uniform(-epsilon, epsilon)
    
            right = phi[(i + 1) % L, j]
            left  = phi[(i - 1) % L, j]
            up    = phi[i, (j + 1) % L]
            down  = phi[i, (j - 1) % L]
    
            old_kin = 0.5 * (
                (right - old_val)**2 +
                (old_val - left)**2 +
                (up - old_val)**2 +
                (old_val - down)**2
            )
    
            new_kin = 0.5 * (
                (right - new_val)**2 +
                (new_val - left)**2 +
                (up - new_val)**2 +
                (new_val - down)**2
            )
    
            old_V = 0.5*m2*old_val**2 + lam/24*old_val**4
            new_V = 0.5*m2*new_val**2 + lam/24*new_val**4
    
            dS = (new_kin + new_V) - (old_kin + old_V)
    
            if dS <= 0 or np.random.random() < np.exp(-dS):
                phi[i, j] = new_val
                accepted += 1
    return accepted / (L*L)



In [5]:
def zero_momentum_correlator(phi):
    # phi[t, x]

    # spatial average
    Phi = np.mean(phi, axis=1)

    Lt = phi.shape[0]
    C = np.zeros(Lt)

    for dt in range(Lt):
        C[dt] = np.mean(Phi * np.roll(Phi, -dt))

    return C


In [6]:
def correlator(phi, r):
    L = phi.shape[0]
    total = 0.0

    for i in range(L):
        for j in range(L):
            total += phi[i, j] * phi[(i+r) % L, j]

    return total / L**2

In [ ]:
start_timer = time.perf_counter()

N_therm = 5000
N_meas = 20000

# Thermalize
for _ in range(N_therm):
    sweep_metropolis(phi, m2, lam, epsilon)

# Masurements
C_sum = np.zeros(L)
acceptance_sum = 0.0
for _ in range(N_meas):
    acceptance_sum += sweep_metropolis(phi, m2, lam, epsilon)

    C_config = zero_momentum_correlator(phi)
    C_sum += C_config
acceptance_rate = acceptance_sum/N_meas
C = C_sum / N_meas

end_timer = time.perf_counter()

execution_time = end_timer - start_timer


print(C)
print(f"Acceptance: {100 * acceptance_rate:.1f}%")
print(f"Code took {execution_time:.6f} seconds to complete.")

In [ ]:
m_eff = np.arccosh( (C[:-2] + C[2:])/(2*C[1:-1]) )
print(m_eff)

In [ ]:
fit_window = m_eff[0:6]

print("m_R =", np.nanmean(fit_window))
print("(fake) spread =", np.nanstd(fit_window))

Should return a mass of $a \cdot m_{eff} \approx 0.500$ and with $a = 1$ (lattice spacing) should return $\approx 0.500$